# ⚔ Laser Chess

**Objetivo:** Hacer que el láser de tu Esfinge alcance al Faraón enemigo.

### Piezas
| Símbolo | Pieza | Descripción |
|---|---|---|
| ♛ | Faraón | Tu objetivo a proteger. No rota. |
| ⇑⇓⇒⇐ | Esfinge | Dispara el láser. No se mueve. |
| ◤◥◣◢ | Triángulo | Refleja el láser por 2 lados; el resto lo destruye. |
| █ | Bloque | Absorbe y destruye el láser. |
| ╱╲ | Espejo Doble | Refleja el láser siempre. Puede intercambiar posición. |

### Comandos
```
f,c-f,c     → Mover pieza a casilla adyacente
R f,c I/D   → Rotar pieza (I=izquierda, D=derecha)
I f,c-f,c   → Intercambiar Espejo Doble con pieza adyacente de menor rango
L           → Ver todos los movimientos legales
A           → Ayuda
Q           → Salir
```

## Motor del juego

In [ ]:
from dataclasses import dataclass
from enum import Enum
import re
from IPython.display import clear_output


# ── Códigos de color ANSI ─────────────────────────────────────────────────────

class Color:
    ROJO     = '\033[91m'
    AZUL     = '\033[96m'
    GRIS     = '\033[90m'
    RESET    = '\033[0m'
    BLANCO   = '\033[97m'
    AMARILLO = '\033[93m'


# ── Enumeraciones ─────────────────────────────────────────────────────────────

class Jugador(str, Enum):
    ROJO = 'ROJO'
    AZUL = 'AZUL'

    @property
    def contrincante(self):
        return Jugador.AZUL if self == Jugador.ROJO else Jugador.ROJO

    @property
    def color(self):
        return Color.ROJO if self == Jugador.ROJO else Color.AZUL


class Direccion(int, Enum):
    NORTE = 0
    ESTE  = 1
    SUR   = 2
    OESTE = 3

    def gira_izq(self): return Direccion((int(self) - 1) % 4)
    def gira_der(self): return Direccion((int(self) + 1) % 4)

    @property
    def delta(self):
        return {
            Direccion.NORTE: (-1,  0),
            Direccion.ESTE:  ( 0,  1),
            Direccion.SUR:   ( 1,  0),
            Direccion.OESTE: ( 0, -1),
        }[self]

    @property
    def opuesta(self):
        return {
            Direccion.NORTE: Direccion.SUR,
            Direccion.SUR:   Direccion.NORTE,
            Direccion.ESTE:  Direccion.OESTE,
            Direccion.OESTE: Direccion.ESTE,
        }[self]


class Esquina(str, Enum):
    NO = 'NO'
    NE = 'NE'
    SE = 'SE'
    SO = 'SO'

    @property
    def diagonal(self):
        return '/' if self in (Esquina.NO, Esquina.SE) else '\\'

    def gira_izq(self):
        orden = [Esquina.NO, Esquina.NE, Esquina.SE, Esquina.SO]
        return orden[(orden.index(self) - 1) % 4]

    def gira_der(self):
        orden = [Esquina.NO, Esquina.NE, Esquina.SE, Esquina.SO]
        return orden[(orden.index(self) + 1) % 4]


class TipoPieza(str, Enum):
    FARAON       = 'K'
    ESFINGE      = 'S'
    TRIANGULO    = 'T'
    BLOQUE       = 'B'
    ESPEJO_DOBLE = 'M'


# ── Constantes del tablero (requieren TipoPieza) ──────────────────────────────

FILAS    = 8
COLUMNAS = 10

CASILLAS_SOLO_ROJO = {(0, 8), (7, 8), *{(r, 0) for r in range(1, 8)}}
CASILLAS_SOLO_AZUL = {(0, 1), (7, 1), *{(r, 9) for r in range(0, 7)}}

# Jerarquía de rango para la mecánica de intercambio del Espejo Doble
RANGO = {
    TipoPieza.TRIANGULO    : 1,
    TipoPieza.BLOQUE       : 2,
    TipoPieza.FARAON       : 3,
    TipoPieza.ESPEJO_DOBLE : 4,
    TipoPieza.ESFINGE      : 5,
}


# ── Pieza (inmutable) ─────────────────────────────────────────────────────────

@dataclass(frozen=True)
class Pieza:
    dueno:        Jugador
    tipo:         TipoPieza
    direccion:    Direccion = Direccion.NORTE
    forma_espejo: str       = '/'
    esquina:      Esquina   = Esquina.NO

    def simbolo(self) -> str:
        if self.tipo == TipoPieza.ESFINGE:
            return {
                Direccion.NORTE: '⇑', Direccion.ESTE: '⇒',
                Direccion.SUR:   '⇓', Direccion.OESTE: '⇐',
            }[self.direccion]
        if self.tipo == TipoPieza.TRIANGULO:
            return {
                Esquina.NO: '◤', Esquina.NE: '◥',
                Esquina.SE: '◢', Esquina.SO: '◣',
            }[self.esquina]
        if self.tipo == TipoPieza.ESPEJO_DOBLE:
            return '╱' if self.forma_espejo == '/' else '╲'
        return {'K': '♛', 'B': '█'}[self.tipo.value]

    def puede_moverse(self) -> bool:
        return self.tipo != TipoPieza.ESFINGE

    def puede_rotar(self) -> bool:
        return self.tipo in {TipoPieza.ESFINGE, TipoPieza.TRIANGULO, TipoPieza.ESPEJO_DOBLE}

    def rota(self, lado: str) -> 'Pieza':
        lado = lado.upper()
        if lado not in {'I', 'D'}:
            raise ValueError('Lado debe ser I (izquierda) o D (derecha)')
        if not self.puede_rotar():
            raise ValueError('Esta pieza no puede rotar')
        if self.tipo == TipoPieza.ESFINGE:
            nueva_dir = self.direccion.gira_izq() if lado == 'I' else self.direccion.gira_der()
            return Pieza(self.dueno, self.tipo, nueva_dir, self.forma_espejo, self.esquina)
        if self.tipo == TipoPieza.TRIANGULO:
            nueva_esq = self.esquina.gira_izq() if lado == 'I' else self.esquina.gira_der()
            return Pieza(self.dueno, self.tipo, self.direccion, nueva_esq.diagonal, nueva_esq)
        # Espejo Doble: alterna entre / y \
        nueva_forma = '\\' if self.forma_espejo == '/' else '/'
        return Pieza(self.dueno, self.tipo, self.direccion, nueva_forma, self.esquina)


# ── Juego principal ───────────────────────────────────────────────────────────

class JuegoLaserChess:

    def __init__(self):
        self.tablero        = [[None] * COLUMNAS for _ in range(FILAS)]
        self.jugador_actual = Jugador.ROJO
        self.ganador        = None
        self.ultimo_mensaje = 'Turno de ROJO — ¡Mueve o rota una pieza!'
        self.historial      = []
        self._setup()

    # ── Configuración inicial ─────────────────────────────────────────────────

    def _setup(self):
        piezas_iniciales = {
            # ROJO
            (0, 0): Pieza(Jugador.ROJO, TipoPieza.ESFINGE,      Direccion.SUR),
            (0, 4): Pieza(Jugador.ROJO, TipoPieza.BLOQUE),
            (0, 5): Pieza(Jugador.ROJO, TipoPieza.FARAON),
            (0, 6): Pieza(Jugador.ROJO, TipoPieza.BLOQUE),
            (0, 7): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.NO, forma_espejo=Esquina.NO.diagonal),
            (1, 2): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.NE, forma_espejo=Esquina.NE.diagonal),
            (3, 0): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.SO, forma_espejo=Esquina.SO.diagonal),
            (3, 4): Pieza(Jugador.ROJO, TipoPieza.ESPEJO_DOBLE, forma_espejo='\\'),
            (3, 5): Pieza(Jugador.ROJO, TipoPieza.ESPEJO_DOBLE, forma_espejo='/'),
            (3, 7): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.NO, forma_espejo=Esquina.NO.diagonal),
            (4, 0): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.NO, forma_espejo=Esquina.NO.diagonal),
            (4, 7): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.SO, forma_espejo=Esquina.SO.diagonal),
            (5, 6): Pieza(Jugador.ROJO, TipoPieza.TRIANGULO,    esquina=Esquina.NO, forma_espejo=Esquina.NO.diagonal),
            # AZUL
            (2, 3): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.SE, forma_espejo=Esquina.SE.diagonal),
            (3, 2): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.NE, forma_espejo=Esquina.NE.diagonal),
            (3, 9): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.SE, forma_espejo=Esquina.SE.diagonal),
            (4, 2): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.SE, forma_espejo=Esquina.SE.diagonal),
            (4, 4): Pieza(Jugador.AZUL, TipoPieza.ESPEJO_DOBLE, forma_espejo='/'),
            (4, 5): Pieza(Jugador.AZUL, TipoPieza.ESPEJO_DOBLE, forma_espejo='\\'),
            (4, 9): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.NE, forma_espejo=Esquina.NE.diagonal),
            (6, 7): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.SO, forma_espejo=Esquina.SO.diagonal),
            (7, 2): Pieza(Jugador.AZUL, TipoPieza.TRIANGULO,    esquina=Esquina.SE, forma_espejo=Esquina.SE.diagonal),
            (7, 3): Pieza(Jugador.AZUL, TipoPieza.BLOQUE),
            (7, 4): Pieza(Jugador.AZUL, TipoPieza.FARAON),
            (7, 5): Pieza(Jugador.AZUL, TipoPieza.BLOQUE),
            (7, 9): Pieza(Jugador.AZUL, TipoPieza.ESFINGE,      Direccion.NORTE),
        }
        for (f, c), p in piezas_iniciales.items():
            self.tablero[f][c] = p

    # ── Helpers de geometría ──────────────────────────────────────────────────

    def _dentro(self, f: int, c: int) -> bool:
        return 0 <= f < FILAS and 0 <= c < COLUMNAS

    def _es_adyacente(self, f1, c1, f2, c2) -> bool:
        return abs(f1 - f2) <= 1 and abs(c1 - c2) <= 1 and (f1, c1) != (f2, c2)

    def _puede_entrar(self, pieza: Pieza, f: int, c: int) -> bool:
        if pieza.dueno == Jugador.ROJO and (f, c) in CASILLAS_SOLO_AZUL:
            return False
        if pieza.dueno == Jugador.AZUL and (f, c) in CASILLAS_SOLO_ROJO:
            return False
        return True

    def _adyacentes(self, f: int, c: int):
        for df in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if df == 0 and dc == 0:
                    continue
                nf, nc = f + df, c + dc
                if self._dentro(nf, nc):
                    yield nf, nc

    # ── Lógica del láser ──────────────────────────────────────────────────────

    def _refleja(self, d: Direccion, forma: str) -> Direccion:
        tabla = {
            '/':  {
                Direccion.NORTE: Direccion.ESTE,  Direccion.SUR:   Direccion.OESTE,
                Direccion.ESTE:  Direccion.NORTE, Direccion.OESTE: Direccion.SUR,
            },
            '\\': {
                Direccion.NORTE: Direccion.OESTE, Direccion.SUR:   Direccion.ESTE,
                Direccion.ESTE:  Direccion.SUR,   Direccion.OESTE: Direccion.NORTE,
            },
        }
        return tabla[forma][d]

    def _es_lado_espejo(self, d: Direccion, esquina: Esquina) -> bool:
        lados_espejo = {
            Esquina.NO: {Direccion.SUR,   Direccion.ESTE},
            Esquina.NE: {Direccion.SUR,   Direccion.OESTE},
            Esquina.SO: {Direccion.NORTE, Direccion.ESTE},
            Esquina.SE: {Direccion.NORTE, Direccion.OESTE},
        }
        return d.opuesta in lados_espejo[esquina]

    def _termina_turno(self, mensaje: str):
        self.ultimo_mensaje = mensaje
        self.jugador_actual = self.jugador_actual.contrincante

    def _dispara_laser(self):
        # Localizar la Esfinge del jugador actual
        pos = next(
            ((f, c) for f in range(FILAS) for c in range(COLUMNAS)
             if self.tablero[f][c]
             and self.tablero[f][c].dueno == self.jugador_actual
             and self.tablero[f][c].tipo  == TipoPieza.ESFINGE),
            None,
        )
        if not pos:
            self.ganador        = self.jugador_actual.contrincante
            self.ultimo_mensaje = f'Esfinge destruida — GANA {self.ganador.value}'
            return

        f, c = pos
        d    = self.tablero[f][c].direccion
        f, c = f + d.delta[0], c + d.delta[1]

        visitados: set = set()   # estados (fila, col, dir) para detectar ciclos

        while self._dentro(f, c):
            estado = (f, c, d)
            if estado in visitados:
                # CORRECCIÓN 3: ciclo detectado — el láser se disipa sin efecto
                self._termina_turno(
                    f'Láser atrapado en ciclo — se disipa. '
                    f'Turno de {self.jugador_actual.contrincante.value}'
                )
                return
            visitados.add(estado)

            q = self.tablero[f][c]
            if q is None:
                f, c = f + d.delta[0], c + d.delta[1]
                continue

            # Espejo Doble: siempre refleja el láser
            if q.tipo == TipoPieza.ESPEJO_DOBLE:
                d    = self._refleja(d, q.forma_espejo)
                f, c = f + d.delta[0], c + d.delta[1]
                continue

            # Triángulo: refleja por sus 2 lados espejo; el resto lo destruye
            if q.tipo == TipoPieza.TRIANGULO:
                if self._es_lado_espejo(d, q.esquina):
                    d    = self._refleja(d, q.forma_espejo)
                    f, c = f + d.delta[0], c + d.delta[1]
                    continue
                else:
                    self.tablero[f][c] = None
                    self._termina_turno(
                        f'Triángulo de {q.dueno.value} destruido — '
                        f'turno de {self.jugador_actual.contrincante.value}'
                    )
                    return

            # Esfinge: absorbe el láser sin destruirse
            if q.tipo == TipoPieza.ESFINGE:
                self._termina_turno(
                    f'Láser absorbido por Esfinge — '
                    f'turno de {self.jugador_actual.contrincante.value}'
                )
                return

            # CORRECCIÓN 1: el ganador es el rival del dueño del Faraón destruido
            if q.tipo == TipoPieza.FARAON:
                self.ganador        = q.dueno.contrincante
                self.ultimo_mensaje = (
                    f'¡FARAÓN DE {q.dueno.value} ALCANZADO! — GANA {self.ganador.value}'
                )
                return

            # CORRECCIÓN 2: el Bloque detiene el láser pero NO se destruye
            if q.tipo == TipoPieza.BLOQUE:
                self._termina_turno(
                    f'Láser bloqueado por Bloque de {q.dueno.value} — '
                    f'turno de {self.jugador_actual.contrincante.value}'
                )
                return

        # El láser salió del tablero sin impactar nada
        self._termina_turno(
            f'Láser salió del tablero — turno de {self.jugador_actual.contrincante.value}'
        )

    # ── Acciones del jugador ──────────────────────────────────────────────────

    def mover(self, f: int, c: int, nf: int, nc: int):
        if self.ganador:
            raise ValueError('El juego ya terminó')
        if not self._dentro(f, c) or not self._dentro(nf, nc):
            raise ValueError('Posición fuera del tablero')
        p = self.tablero[f][c]
        if not p:
            raise ValueError(f'No hay pieza en ({f},{c})')
        if p.dueno != self.jugador_actual:
            raise ValueError(f'Esa pieza es de {p.dueno.value}')
        if not p.puede_moverse():
            raise ValueError('La Esfinge no puede moverse')
        if self.tablero[nf][nc] is not None:
            raise ValueError('Esa casilla está ocupada')
        if not self._puede_entrar(p, nf, nc):
            raise ValueError('Casilla restringida al color contrario')
        if not self._es_adyacente(f, c, nf, nc):
            raise ValueError('Solo puedes mover a una casilla adyacente')
        self.historial.append(f'MOVER ({f},{c})→({nf},{nc})')
        self.tablero[nf][nc] = p
        self.tablero[f][c]   = None
        self._dispara_laser()

    def rotar(self, f: int, c: int, lado: str):
        if self.ganador:
            raise ValueError('El juego ya terminó')
        if not self._dentro(f, c):
            raise ValueError('Posición fuera del tablero')
        p = self.tablero[f][c]
        if not p:
            raise ValueError(f'No hay pieza en ({f},{c})')
        if p.dueno != self.jugador_actual:
            raise ValueError(f'Esa pieza es de {p.dueno.value}')
        if not p.puede_rotar():
            raise ValueError('El Faraón y el Bloque no pueden rotar')
        np_ = p.rota(lado)
        if p.tipo == TipoPieza.ESFINGE:
            df, dc = np_.direccion.delta
            if not self._dentro(f + df, c + dc):
                raise ValueError('La Esfinge no puede apuntar fuera del tablero')
        self.historial.append(f'ROTAR ({f},{c}) {lado}')
        self.tablero[f][c] = np_
        self._dispara_laser()

    def intercambiar(self, f: int, c: int, nf: int, nc: int):
        if self.ganador:
            raise ValueError('El juego ya terminó')
        if not self._dentro(f, c) or not self._dentro(nf, nc):
            raise ValueError('Fuera del tablero')
        p, q = self.tablero[f][c], self.tablero[nf][nc]
        if not p or p.dueno != self.jugador_actual:
            raise ValueError('No es tu pieza')
        if not q:
            raise ValueError('No hay pieza destino')
        if p.tipo != TipoPieza.ESPEJO_DOBLE:
            raise ValueError('Solo el Espejo Doble puede intercambiar')
        if q.tipo in (TipoPieza.FARAON, TipoPieza.ESFINGE):
            raise ValueError('No puedes intercambiar con Faraón o Esfinge')
        if RANGO[q.tipo] >= RANGO[p.tipo]:
            raise ValueError('Solo puedes intercambiar con una pieza de menor rango')
        if not self._puede_entrar(p, nf, nc) or not self._puede_entrar(q, f, c):
            raise ValueError('Intercambio restringido por zona')
        if not self._es_adyacente(f, c, nf, nc):
            raise ValueError('Solo puedes intercambiar con una pieza adyacente')
        self.historial.append(f'INTERCAMBIAR ({f},{c})↔({nf},{nc})')
        self.tablero[f][c], self.tablero[nf][nc] = q, p
        self._dispara_laser()

    # ── Parseo de comandos de texto ───────────────────────────────────────────

    def procesa_comando(self, cmd: str):
        cmd = cmd.strip().upper()
        if not cmd:
            raise ValueError('Comando vacío')
        if cmd == 'L':
            return self.muestra_legales()
        if cmd == 'A':
            return self.muestra_ayuda()
        if cmd == 'Q':
            raise KeyboardInterrupt()

        if cmd.startswith('I '):
            m = re.match(r'I\s+(\d+),(\d+)-(\d+),(\d+)', cmd)
            if not m:
                raise ValueError('Formato incorrecto — usa: I f,c-f,c')
            return self.intercambiar(*map(int, m.groups()))

        if cmd.startswith('R '):
            partes = cmd.split()
            if len(partes) < 3:
                raise ValueError('Formato incorrecto — usa: R f,c I/D')
            m = re.match(r'(\d+),(\d+)', partes[1])
            if not m:
                raise ValueError('Formato incorrecto — usa: R f,c I/D')
            f, c = map(int, m.groups())
            return self.rotar(f, c, partes[2])

        if '-' in cmd and ',' in cmd:
            m = re.match(r'(\d+),(\d+)-(\d+),(\d+)', cmd)
            if not m:
                raise ValueError('Formato incorrecto — usa: f,c-f,c')
            return self.mover(*map(int, m.groups()))

        raise ValueError('Comando desconocido. Escribe A para ver la ayuda.')

    # ── Visualización ─────────────────────────────────────────────────────────

    def muestra_tablero(self, error: str = None):
        w  = COLUMNAS * 2 + 3
        eq = '═' * w

        print(f"{Color.BLANCO}╔{eq}╗{Color.RESET}")
        print(f"{Color.BLANCO}║  ⚔  LASER CHESS  ⚔  {' ' * (w - 22)}║{Color.RESET}")

        turno_str = f"TURNO: {self.jugador_actual.color}{self.jugador_actual.value}{Color.RESET}"
        padding   = w - 9 - len(self.jugador_actual.value)
        print(f"{Color.BLANCO}║  {turno_str}{Color.BLANCO}{' ' * padding}║{Color.RESET}")

        msg = self.ultimo_mensaje[:w - 3]
        print(f"{Color.BLANCO}║  {msg:<{w - 2}} ║{Color.RESET}")

        if error:
            emsg = f"⚠  {error}"[:w - 3]
            print(f"{Color.ROJO}║  {emsg:<{w - 2}} ║{Color.RESET}")

        print(f"{Color.BLANCO}╚{eq}╝{Color.RESET}\n")

        print(f"{Color.GRIS}    " + "  ".join(str(i) for i in range(COLUMNAS)) + Color.RESET)
        print(f"{Color.GRIS}   ╔" + "══" * COLUMNAS + f"╗{Color.RESET}")
        for fila in range(FILAS):
            linea = f"{Color.GRIS}{fila} ║{Color.RESET}"
            for col in range(COLUMNAS):
                p = self.tablero[fila][col]
                if p is None:
                    if   (fila, col) in CASILLAS_SOLO_ROJO: linea += f"{Color.ROJO}▨ {Color.RESET}"
                    elif (fila, col) in CASILLAS_SOLO_AZUL: linea += f"{Color.AZUL}▨ {Color.RESET}"
                    else:                                    linea += f"{Color.GRIS}· {Color.RESET}"
                else:
                    linea += f"{p.dueno.color}{p.simbolo()} {Color.RESET}"
            print(linea + f"{Color.GRIS}║{Color.RESET}")
        print(f"{Color.GRIS}   ╚" + "══" * COLUMNAS + f"╝{Color.RESET}")

        print(f"""
{Color.GRIS}Leyenda:{Color.RESET}
  {Color.ROJO}ROJO:{Color.RESET} ♛=Faraón  ⇓=Esfinge  ◤◥◣◢=Triángulo  █=Bloque  ╱╲=Espejo
  {Color.AZUL}AZUL:{Color.RESET} ♛=Faraón  ⇑=Esfinge  ◤◥◣◢=Triángulo  █=Bloque  ╱╲=Espejo
  {Color.GRIS}▨=Casilla exclusiva | Comandos: f,c-f,c · R f,c I/D · I f,c-f,c · L · A · Q{Color.RESET}
""")

    def muestra_legales(self):
        movimientos = []
        for f in range(FILAS):
            for c in range(COLUMNAS):
                p = self.tablero[f][c]
                if not p or p.dueno != self.jugador_actual:
                    continue
                if p.puede_rotar():
                    for lado, etiq in (('I', 'rotar izq'), ('D', 'rotar der')):
                        if p.tipo == TipoPieza.ESFINGE:
                            nueva_dir = p.rota(lado).direccion
                            df2, dc2 = nueva_dir.delta
                            if not self._dentro(f + df2, c + dc2):
                                continue
                        movimientos.append(f'R {f},{c} {lado}  ({p.simbolo()} {etiq})')
                if p.puede_moverse():
                    for nf, nc in self._adyacentes(f, c):
                        if self.tablero[nf][nc] is None and self._puede_entrar(p, nf, nc):
                            movimientos.append(f'{f},{c}-{nf},{nc}  (mover {p.simbolo()})')
                if p.tipo == TipoPieza.ESPEJO_DOBLE:
                    for nf, nc in self._adyacentes(f, c):
                        q = self.tablero[nf][nc]
                        if (q
                                and RANGO[q.tipo] < RANGO[p.tipo]
                                and q.tipo not in (TipoPieza.FARAON, TipoPieza.ESFINGE)
                                and self._puede_entrar(q, f, c)
                                and self._puede_entrar(p, nf, nc)):
                            movimientos.append(
                                f'I {f},{c}-{nf},{nc}  (intercambiar {p.simbolo()}↔{q.simbolo()})'
                            )

        limite = 50
        print(f"\n{Color.AMARILLO}MOVIMIENTOS LEGALES ({len(movimientos)}):{Color.RESET}")
        for mv in (movimientos[:limite] if movimientos else ['Ninguno disponible']):
            print(f'  {mv}')
        if len(movimientos) > limite:
            print(f'  ... y {len(movimientos) - limite} más')
        print()

    def muestra_ayuda(self):
        print(f"""
{Color.BLANCO}╔══════════════════════════════════════════╗
║           COMANDOS DEL JUEGO             ║
╠══════════════════════════════════════════╣
║  f,c-f,c     Mover pieza (adyacente)    ║
║  R f,c I/D   Rotar pieza I=izq D=der    ║
║  I f,c-f,c   Intercambiar (espejo doble)║
║  L           Ver movimientos legales     ║
║  A           Ver esta ayuda              ║
║  Q           Salir del juego             ║
╠══════════════════════════════════════════╣
║  PIEZAS:                                 ║
║  ♛ Faraón   — objetivo a proteger       ║
║  ⇑⇓⇒⇐ Esfinge — dispara el láser       ║
║  ◤◥◣◢ Triángulo — refleja 2 caras       ║
║  █ Bloque   — detiene y absorbe el láser║
║  ╱╲ Espejo doble — refleja siempre      ║
╚══════════════════════════════════════════╝{Color.RESET}
""")




## Motor de Agentes Inteligentes

In [ ]:
import copy
import time
import random
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional


# ── 1. GENERADOR DE MOVIMIENTOS ───────────────────────────────────────────────

def genera_movimientos(juego: JuegoLaserChess) -> list:
    movs = []
    jug  = juego.jugador_actual
    for f in range(FILAS):
        for c in range(COLUMNAS):
            p = juego.tablero[f][c]
            if not p or p.dueno != jug:
                continue
            if p.puede_rotar():
                for lado in ('I', 'D'):
                    if p.tipo == TipoPieza.ESFINGE:
                        nueva_dir = p.rota(lado).direccion
                        df2, dc2 = nueva_dir.delta
                        if not juego._dentro(f + df2, c + dc2):
                            continue
                    movs.append(f'R {f},{c} {lado}')
            if p.puede_moverse():
                for nf, nc in juego._adyacentes(f, c):
                    if juego.tablero[nf][nc] is None and juego._puede_entrar(p, nf, nc):
                        movs.append(f'{f},{c}-{nf},{nc}')
            if p.tipo == TipoPieza.ESPEJO_DOBLE:
                for nf, nc in juego._adyacentes(f, c):
                    q = juego.tablero[nf][nc]
                    if (q
                            and RANGO[q.tipo] < RANGO[p.tipo]
                            and q.tipo not in (TipoPieza.FARAON, TipoPieza.ESFINGE)
                            and juego._puede_entrar(q, f, c)
                            and juego._puede_entrar(p, nf, nc)):
                        movs.append(f'I {f},{c}-{nf},{nc}')
    return movs


def aplica_movimiento(juego: JuegoLaserChess, cmd: str) -> JuegoLaserChess:
    nuevo = copy.deepcopy(juego)
    nuevo.procesa_comando(cmd)
    return nuevo


# ── 2. PUNTUACIÓN DE MATERIAL ─────────────────────────────────────────────────

_VALOR_PIEZA = {
    TipoPieza.TRIANGULO    : 2,
    TipoPieza.BLOQUE       : 3,
    TipoPieza.ESPEJO_DOBLE : 5,
    TipoPieza.ESFINGE      : 10,
    TipoPieza.FARAON       : 0,
}


def calcular_puntos(tablero: list, jugador: Jugador) -> int:
    """Suma los valores de todas las piezas vivas de un jugador (métrica de puntos finales)."""
    return sum(
        _VALOR_PIEZA.get(p.tipo, 0)
        for fila in tablero
        for p in fila
        if p and p.dueno == jugador
    )


# ── 3. HEURÍSTICAS DE EVALUACIÓN ─────────────────────────────────────────────

_CASILLAS_CENTRO = frozenset((f, c) for f in range(2, 6) for c in range(3, 7))


def h1_material(juego: JuegoLaserChess, jugador: Jugador) -> float:
    """H1 — Material: suma de valores de piezas propias menos piezas rivales."""
    score = 0.0
    for fila in juego.tablero:
        for p in fila:
            if p is None:
                continue
            v = _VALOR_PIEZA.get(p.tipo, 0)
            score += v if p.dueno == jugador else -v
    return score


def h2_control_centro(juego: JuegoLaserChess, jugador: Jugador) -> float:
    """H2 — Control del centro: piezas en zona central (filas 2-5, cols 3-6)."""
    score = 0.0
    for (f, c) in _CASILLAS_CENTRO:
        p = juego.tablero[f][c]
        if p is not None:
            score += 1.0 if p.dueno == jugador else -1.0
    return score


def h3_movilidad(juego: JuegoLaserChess, jugador: Jugador) -> float:
    """H3 — Movilidad: diferencia en acciones disponibles (propias - rivales)."""
    prop = 0
    rival_cnt = 0
    for f in range(FILAS):
        for c in range(COLUMNAS):
            p = juego.tablero[f][c]
            if p is None:
                continue
            n = 0
            if p.puede_rotar():
                n += 2
            if p.puede_moverse():
                n += sum(
                    1 for nf, nc in juego._adyacentes(f, c)
                    if juego.tablero[nf][nc] is None and juego._puede_entrar(p, nf, nc)
                )
            if p.dueno == jugador:
                prop += n
            else:
                rival_cnt += n
    return float(prop - rival_cnt)


def h4_seguridad_faraon(juego: JuegoLaserChess, jugador: Jugador) -> float:
    """H4 — Seguridad del Faraón: Bloques/Triángulos propios adyacentes al Faraón."""
    faraon_pos = None
    for f in range(FILAS):
        for c in range(COLUMNAS):
            p = juego.tablero[f][c]
            if p and p.tipo == TipoPieza.FARAON and p.dueno == jugador:
                faraon_pos = (f, c)
                break
        if faraon_pos:
            break
    if not faraon_pos:
        return -100.0
    ff, fc = faraon_pos
    protectores = 0
    for df in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if df == 0 and dc == 0:
                continue
            nf, nc = ff + df, fc + dc
            if 0 <= nf < FILAS and 0 <= nc < COLUMNAS:
                p = juego.tablero[nf][nc]
                if p and p.dueno == jugador and p.tipo in (TipoPieza.BLOQUE, TipoPieza.TRIANGULO):
                    protectores += 1
    return float(protectores)


def h5_amenaza_faraon_rival(juego: JuegoLaserChess, jugador: Jugador) -> float:
    """H5 — Amenaza al Faraón rival: distancia mínima del láser propio al Faraón rival."""
    rival        = jugador.contrincante
    faraon_rival = None
    esfinge_pos  = None
    for f in range(FILAS):
        for c in range(COLUMNAS):
            p = juego.tablero[f][c]
            if p is None:
                continue
            if p.tipo == TipoPieza.FARAON  and p.dueno == rival:
                faraon_rival = (f, c)
            if p.tipo == TipoPieza.ESFINGE and p.dueno == jugador:
                esfinge_pos  = (f, c)
    # FIX: si el Faraón rival ya no existe, el nodo es ganado → valor máximo
    if not faraon_rival:
        return float('inf')
    if not esfinge_pos:
        return -50.0
    ff, fc = esfinge_pos
    d      = juego.tablero[ff][fc].direccion
    lf, lc = ff + d.delta[0], fc + d.delta[1]
    min_dist: float = float('inf')
    visitados: set  = set()
    while juego._dentro(lf, lc):
        estado = (lf, lc, d)
        if estado in visitados:
            break
        visitados.add(estado)
        dist = abs(lf - faraon_rival[0]) + abs(lc - faraon_rival[1])
        if dist < min_dist:
            min_dist = dist
        if min_dist == 0:
            break
        q = juego.tablero[lf][lc]
        if q is None:
            lf, lc = lf + d.delta[0], lc + d.delta[1]
            continue
        if q.tipo == TipoPieza.ESPEJO_DOBLE:
            d    = juego._refleja(d, q.forma_espejo)
            lf, lc = lf + d.delta[0], lc + d.delta[1]
            continue
        if q.tipo == TipoPieza.TRIANGULO and juego._es_lado_espejo(d, q.esquina):
            d    = juego._refleja(d, q.forma_espejo)
            lf, lc = lf + d.delta[0], lc + d.delta[1]
            continue
        break
    if min_dist == float('inf'):
        min_dist = 15.0
    return max(0.0, 10.0 - float(min_dist))


HEURISTICAS_DISPONIBLES: dict = {
    1: ('H1 — Material',                h1_material),
    2: ('H2 — Control del centro',      h2_control_centro),
    3: ('H3 — Movilidad',               h3_movilidad),
    4: ('H4 — Seguridad del Faraón',    h4_seguridad_faraon),
    5: ('H5 — Amenaza al Faraón rival', h5_amenaza_faraon_rival),
}

PESOS_CONFIG_1: dict = {1: 1.5, 2: 0.5, 3: 0.3, 4: 1.2, 5: 1.8}   # Equilibrada
PESOS_CONFIG_2: dict = {1: 0.5, 2: 0.3, 3: 0.2, 4: 0.6, 5: 3.5}   # Agresiva


@dataclass
class ConfiguracionHeuristica:
    activas: set  = field(default_factory=lambda: {1, 2, 3, 4, 5})
    pesos:   dict = field(default_factory=lambda: dict(PESOS_CONFIG_1))

    def evaluar(self, juego: JuegoLaserChess, jugador: Jugador) -> float:
        total = 0.0
        for idx, (_nombre, fn) in HEURISTICAS_DISPONIBLES.items():
            if idx in self.activas:
                peso = self.pesos.get(idx, 1.0)
                if peso:
                    total += peso * fn(juego, jugador)
        return total

    def descripcion(self) -> str:
        lineas = []
        for idx, (nombre, _) in HEURISTICAS_DISPONIBLES.items():
            estado = f'peso={self.pesos.get(idx, 0):.1f}' if idx in self.activas else 'OFF'
            lineas.append(f'  {nombre}: {estado}')
        return '\n'.join(lineas)


# ── 4. MINIMAX CON ALPHA-BETA, IDS, TABLA DE TRANSPOSICIÓN ───────────────────

_INF      = float('inf')
_TABLA_MAX = 50_000   # entradas máximas en la tabla de transposición


class _TiempoAgotado(Exception):
    pass


def _hash_estado(juego: JuegoLaserChess) -> tuple:
    """Hash compacto del estado: tupla de todas las celdas + turno actual."""
    return (
        tuple(juego.tablero[f][c] for f in range(FILAS) for c in range(COLUMNAS)),
        juego.jugador_actual,
    )


def _ordena_movimientos(movs: list, juego: JuegoLaserChess) -> list:
    """
    Ordena movimientos para maximizar la poda Alpha-Beta:
    1. Rotaciones de la Esfinge (cambian el láser directamente)
    2. Otras rotaciones (cambian la ruta del láser)
    3. Intercambios (reposicionamiento táctico)
    4. Movimientos de piezas (menor impacto inmediato)
    """
    esf_pos = None
    for f in range(FILAS):
        for c in range(COLUMNAS):
            p = juego.tablero[f][c]
            if p and p.tipo == TipoPieza.ESFINGE and p.dueno == juego.jugador_actual:
                esf_pos = f'{f},{c}'
                break
        if esf_pos:
            break

    rots_esf, rots_otras, intercs, moves = [], [], [], []
    for m in movs:
        if m.startswith('R '):
            (rots_esf if esf_pos and esf_pos in m else rots_otras).append(m)
        elif m.startswith('I '):
            intercs.append(m)
        else:
            moves.append(m)
    return rots_esf + rots_otras + intercs + moves


def _minimax(
    juego:       JuegoLaserChess,
    profundidad: int,
    alpha:       float,
    beta:        float,
    maximizando: bool,
    jugador_ia:  Jugador,
    config_h:    ConfiguracionHeuristica,
    t_limite:    float,
    t_inicio:    float,
    nodos:       list,
    tabla:       dict,   # tabla de transposición compartida por toda la búsqueda IDS
) -> tuple:
    """
    Minimax con poda Alpha-Beta, tabla de transposición y ordenamiento de movimientos.
    Retorna (valor_float, mejor_cmd | None).
    """
    nodos[0] += 1

    if time.time() - t_inicio >= t_limite:
        raise _TiempoAgotado()

    if juego.ganador is not None:
        return (_INF, None) if juego.ganador == jugador_ia else (-_INF, None)

    # ── Consulta tabla de transposición ──────────────────────────────────────
    alpha_ini = alpha
    beta_ini  = beta
    hash_key  = _hash_estado(juego)
    entrada   = tabla.get(hash_key)
    mejor_mov_tt = None

    if entrada and entrada['prof'] >= profundidad:
        flag, val_tt, mov_tt = entrada['flag'], entrada['val'], entrada['mov']
        if flag == 'exact':
            return (val_tt, mov_tt)
        if flag == 'lower':
            alpha = max(alpha, val_tt)
        elif flag == 'upper':
            beta  = min(beta,  val_tt)
        if alpha >= beta:
            return (val_tt, mov_tt)
        mejor_mov_tt = mov_tt  # movimiento sugerido por la tabla para intentar primero

    movimientos = _ordena_movimientos(genera_movimientos(juego), juego)

    if profundidad == 0 or not movimientos:
        return (config_h.evaluar(juego, jugador_ia), None)

    # Poner el movimiento de la tabla al frente para mejor poda
    if mejor_mov_tt and mejor_mov_tt in movimientos:
        movimientos.remove(mejor_mov_tt)
        movimientos.insert(0, mejor_mov_tt)

    mejor_mov = movimientos[0]

    if maximizando:
        mejor_val = -_INF
        for mov in movimientos:
            hijo   = aplica_movimiento(juego, mov)
            val, _ = _minimax(hijo, profundidad - 1, alpha, beta,
                              False, jugador_ia, config_h, t_limite, t_inicio, nodos, tabla)
            if val > mejor_val:
                mejor_val = val
                mejor_mov = mov
            alpha = max(alpha, mejor_val)
            if beta <= alpha:
                break
    else:
        mejor_val = _INF
        for mov in movimientos:
            hijo   = aplica_movimiento(juego, mov)
            val, _ = _minimax(hijo, profundidad - 1, alpha, beta,
                              True, jugador_ia, config_h, t_limite, t_inicio, nodos, tabla)
            if val < mejor_val:
                mejor_val = val
                mejor_mov = mov
            beta = min(beta, mejor_val)
            if beta <= alpha:
                break

    # ── Almacenar en tabla de transposición ──────────────────────────────────
    if len(tabla) < _TABLA_MAX:
        if   mejor_val <= alpha_ini: flag = 'upper'
        elif mejor_val >= beta_ini:  flag = 'lower'
        else:                        flag = 'exact'
        tabla[hash_key] = {'prof': profundidad, 'val': mejor_val,
                           'mov': mejor_mov,    'flag': flag}

    return (mejor_val, mejor_mov)


def minimax_ids(
    juego:           JuegoLaserChess,
    jugador_ia:      Jugador,
    config_h:        ConfiguracionHeuristica,
    t_limite:        float,
    profundidad_max: int = 10,
) -> tuple:
    """
    IDS sobre Minimax Alpha-Beta con tabla de transposición persistente.
    La tabla se comparte entre iteraciones: info de profundidad menor orienta búsquedas mayores.
    Retorna (mejor_cmd, profundidad_alcanzada, nodos_expandidos_total).
    """
    t_inicio       = time.time()
    movs_init      = genera_movimientos(juego)
    if not movs_init:
        return (None, 0, 0)

    mejor_mov      = random.choice(movs_init)
    prof_alcanzada = 0
    nodos          = [0]
    tabla: dict    = {}   # tabla de transposición compartida entre profundidades IDS

    for prof in range(1, profundidad_max + 1):
        if time.time() - t_inicio >= t_limite:
            break
        try:
            val, mov = _minimax(
                juego, prof, -_INF, _INF,
                True, jugador_ia, config_h, t_limite, t_inicio, nodos, tabla,
            )
            if mov is not None:
                mejor_mov      = mov
                prof_alcanzada = prof
            if val >= _INF:
                break
        except _TiempoAgotado:
            break

    return (mejor_mov, prof_alcanzada, nodos[0])


# ── 5. PERFILES DE AGENTE ─────────────────────────────────────────────────────

class Agente(ABC):
    def __init__(self, jugador: Jugador, nombre: str):
        self.jugador = jugador
        self.nombre  = nombre

    @abstractmethod
    def elegir_movimiento(self, juego: JuegoLaserChess) -> Optional[str]: ...

    def __str__(self) -> str:
        return f'{self.nombre} [{self.jugador.value}]'


class AgenteHumano(Agente):
    def __init__(self, jugador: Jugador):
        super().__init__(jugador, 'Humano')
    def elegir_movimiento(self, juego: JuegoLaserChess) -> Optional[str]:
        return None


class AgenteAleatorio(Agente):
    def __init__(self, jugador: Jugador):
        super().__init__(jugador, 'Aleatorio')
    def elegir_movimiento(self, juego: JuegoLaserChess) -> Optional[str]:
        movs = genera_movimientos(juego)
        return random.choice(movs) if movs else None


class AgenteGreedy(Agente):
    def __init__(self, jugador: Jugador, config_h: ConfiguracionHeuristica = None):
        super().__init__(jugador, 'Greedy')
        self.config_h = config_h or ConfiguracionHeuristica()
    def elegir_movimiento(self, juego: JuegoLaserChess) -> Optional[str]:
        movs = genera_movimientos(juego)
        if not movs:
            return None
        mejor_mov = None
        mejor_val = -_INF
        for mov in movs:
            hijo = aplica_movimiento(juego, mov)
            if hijo.ganador == self.jugador:
                return mov
            val = self.config_h.evaluar(hijo, self.jugador)
            if val > mejor_val:
                mejor_val = val
                mejor_mov = mov
        return mejor_mov


class AgentePeorDecision(Agente):
    def __init__(self, jugador: Jugador, config_h: ConfiguracionHeuristica = None):
        super().__init__(jugador, 'PeorDecision')
        self.config_h = config_h or ConfiguracionHeuristica()
    def elegir_movimiento(self, juego: JuegoLaserChess) -> Optional[str]:
        movs = genera_movimientos(juego)
        if not movs:
            return None
        peor_mov = None
        peor_val = _INF
        for mov in movs:
            val = self.config_h.evaluar(aplica_movimiento(juego, mov), self.jugador)
            if val < peor_val:
                peor_val = val
                peor_mov = mov
        return peor_mov


class AgenteMinimax(Agente):
    def __init__(
        self,
        jugador:         Jugador,
        config_h:        ConfiguracionHeuristica = None,
        tiempo_limite:   float = 3.0,
        profundidad_max: int   = 10,
        nombre:          str   = 'Minimax',
    ):
        super().__init__(jugador, nombre)
        self.config_h           = config_h or ConfiguracionHeuristica()
        self.tiempo_limite      = tiempo_limite
        self.profundidad_max    = profundidad_max
        self.ultima_profundidad = 0
        self.ultimo_tiempo_s    = 0.0
        self.ultimo_nodos       = 0
        self._suma_prof: int    = 0   # acumulado de profundidades para calcular promedio
        self._turnos:    int    = 0   # turnos jugados

    @property
    def prof_media(self) -> float:
        """Profundidad promedio de búsqueda a lo largo de toda la partida."""
        return self._suma_prof / self._turnos if self._turnos else 0.0

    def elegir_movimiento(self, juego: JuegoLaserChess) -> Optional[str]:
        t0 = time.time()
        mov, prof, nodos = minimax_ids(
            juego, self.jugador, self.config_h,
            self.tiempo_limite, self.profundidad_max,
        )
        self.ultima_profundidad  = prof
        self.ultimo_tiempo_s     = time.time() - t0
        self.ultimo_nodos        = nodos
        self._suma_prof         += prof
        self._turnos            += 1
        return mov

    def __str__(self) -> str:
        return (f'{self.nombre} [{self.jugador.value}] '
                f'(t={self.tiempo_limite}s, prof_max={self.profundidad_max})')


# ── 6. CONFIGURACIÓN Y CONTROLADOR DE PARTIDAS ───────────────────────────────

@dataclass
class ResultadoPartida:
    ganador:          Optional[Jugador]
    jugadas:          int
    duracion_s:       float
    historial:        list
    puntos_rojo:      int   = 0
    puntos_azul:      int   = 0
    nodos_expandidos: int   = 0
    prof_promedio:    float = 0.0   # profundidad media de búsqueda de los agentes Minimax


@dataclass
class ConfiguracionPartida:
    agente_rojo:     Agente
    agente_azul:     Agente
    num_partidas:    int  = 1
    max_jugadas:     int  = 300
    mostrar_tablero: bool = True
    pausa_ms:        int  = 0


class ControladorPartida:

    def __init__(self, config: ConfiguracionPartida):
        self.config     = config
        self.resultados: list = []

    def _agente_de(self, jugador: Jugador) -> Agente:
        return self.config.agente_rojo if jugador == Jugador.ROJO else self.config.agente_azul

    def _es_humano(self, jugador: Jugador) -> bool:
        return isinstance(self._agente_de(jugador), AgenteHumano)

    def jugar_partida(self) -> ResultadoPartida:
        juego             = JuegoLaserChess()
        error_actual      = None
        t0                = time.time()
        nodos_total_juego = 0

        while not juego.ganador and len(juego.historial) < self.config.max_jugadas:
            agente = self._agente_de(juego.jugador_actual)

            if self.config.mostrar_tablero:
                clear_output(wait=False)
                juego.muestra_tablero(error=error_actual)
                error_actual = None

            if self._es_humano(juego.jugador_actual):
                try:
                    cmd = input(f'[{juego.jugador_actual.value}] > ').strip()
                    if not cmd:
                        continue
                    juego.procesa_comando(cmd)
                except KeyboardInterrupt:
                    print('\nPartida cancelada.')
                    break
                except Exception as e:
                    error_actual = str(e)
                    continue
            else:
                color = juego.jugador_actual.color
                rst   = Color.RESET
                if self.config.mostrar_tablero:
                    print(f'  {color}[{juego.jugador_actual.value}]{rst} — '
                          f'{agente.nombre} pensando...', end='', flush=True)

                cmd = agente.elegir_movimiento(juego)

                if isinstance(agente, AgenteMinimax):
                    nodos_total_juego += agente.ultimo_nodos
                    if self.config.mostrar_tablero:
                        print(f'\r  {color}[{juego.jugador_actual.value}]{rst} — '
                              f'{agente.nombre}: '
                              f'prof={agente.ultima_profundidad} '
                              f'nodos={agente.ultimo_nodos:,} '
                              f't={agente.ultimo_tiempo_s:.2f}s → {cmd}     ')
                elif self.config.mostrar_tablero:
                    print(f'\r  {color}[{juego.jugador_actual.value}]{rst} — '
                          f'{agente.nombre} → {cmd}     ')

                if cmd is None:
                    print(f'\n  {agente} no encontró movimientos.')
                    break
                try:
                    juego.procesa_comando(cmd)
                except Exception as e:
                    error_actual = f'Error IA ({agente.nombre}): {e}'
                    continue

                if self.config.pausa_ms > 0:
                    time.sleep(self.config.pausa_ms / 1000)

        # Calcular puntos y profundidad promedio
        puntos_r  = calcular_puntos(juego.tablero, Jugador.ROJO)
        puntos_a  = calcular_puntos(juego.tablero, Jugador.AZUL)

        # Promedio de profundidad entre todos los agentes Minimax de la partida
        profs: list = []
        for ag in (self.config.agente_rojo, self.config.agente_azul):
            if isinstance(ag, AgenteMinimax) and ag._turnos:
                profs.append(ag.prof_media)
        prof_prom = sum(profs) / len(profs) if profs else 0.0

        if self.config.mostrar_tablero:
            clear_output(wait=False)
            juego.muestra_tablero()
            if juego.ganador:
                g = juego.ganador
                print(f'{g.color}{"═" * 47}{Color.RESET}')
                print(f'{g.color}  GANA LA PARTIDA: {g.value}{Color.RESET}')
                print(f'{g.color}{"═" * 47}{Color.RESET}')
            elif len(juego.historial) >= self.config.max_jugadas:
                print(f'{Color.AMARILLO}  EMPATE — límite de {self.config.max_jugadas} jugadas{Color.RESET}')

            print(f'\n  {Color.ROJO}Puntos ROJO: {puntos_r}{Color.RESET}  |  '
                  f'{Color.AZUL}Puntos AZUL: {puntos_a}{Color.RESET}')
            if nodos_total_juego:
                print(f'  Nodos expandidos (Minimax total): {nodos_total_juego:,}')
            if prof_prom:
                print(f'  Profundidad promedio de búsqueda: {prof_prom:.2f}')

            print(f'\n{Color.GRIS}Historial ({len(juego.historial)} jugadas):{Color.RESET}')
            for i, mov in enumerate(juego.historial, 1):
                c = Color.ROJO if i % 2 == 1 else Color.AZUL
                n = 'ROJO'     if i % 2 == 1 else 'AZUL'
                print(f'  {c}{i:3d}. {n}: {mov}{Color.RESET}')

        return ResultadoPartida(
            ganador          = juego.ganador,
            jugadas          = len(juego.historial),
            duracion_s       = time.time() - t0,
            historial        = list(juego.historial),
            puntos_rojo      = puntos_r,
            puntos_azul      = puntos_a,
            nodos_expandidos = nodos_total_juego,
            prof_promedio    = prof_prom,
        )

    def ejecutar(self) -> list:
        self.resultados = []
        victorias = {Jugador.ROJO: 0, Jugador.AZUL: 0, None: 0}

        for i in range(self.config.num_partidas):
            if self.config.num_partidas > 1:
                print(f'\n{Color.BLANCO}{"─" * 50}')
                print(f'  PARTIDA {i + 1} / {self.config.num_partidas}')
                print(f'{"─" * 50}{Color.RESET}')

            resultado = self.jugar_partida()
            self.resultados.append(resultado)
            victorias[resultado.ganador] += 1

            if self.config.num_partidas > 1 and not self.config.mostrar_tablero:
                g      = resultado.ganador
                estado = 'EMPATE' if g is None else f'{g.value} gana'
                print(f'  {estado} | {resultado.jugadas} jug | '
                      f'pts R={resultado.puntos_rojo} A={resultado.puntos_azul} | '
                      f'nodos={resultado.nodos_expandidos:,} | '
                      f'prof_prom={resultado.prof_promedio:.1f} | '
                      f'{resultado.duracion_s:.1f}s')

        if self.config.num_partidas > 1:
            total   = self.config.num_partidas
            t_total = sum(r.duracion_s       for r in self.resultados)
            j_prom  = sum(r.jugadas          for r in self.resultados) / total
            n_prom  = sum(r.nodos_expandidos for r in self.resultados) / total
            pr_prom = sum(r.puntos_rojo      for r in self.resultados) / total
            pa_prom = sum(r.puntos_azul      for r in self.resultados) / total
            pp_prom = sum(r.prof_promedio    for r in self.resultados) / total
            rojo    = self.config.agente_rojo
            azul    = self.config.agente_azul

            print(f'\n{Color.BLANCO}{"═" * 52}')
            print(f'  RESUMEN — {total} PARTIDAS')
            print(f'{"═" * 52}{Color.RESET}')
            print(f'  {Color.ROJO}{rojo.nombre}{Color.RESET}: '
                  f'{victorias[Jugador.ROJO]} victorias '
                  f'({victorias[Jugador.ROJO]/total*100:.0f}%)')
            print(f'  {Color.AZUL}{azul.nombre}{Color.RESET}: '
                  f'{victorias[Jugador.AZUL]} victorias '
                  f'({victorias[Jugador.AZUL]/total*100:.0f}%)')
            print(f'  Empates/Sin resultado: {victorias[None]}')
            print(f'  Prom. jugadas:         {j_prom:.1f}')
            print(f'  Prom. puntos ROJO:     {pr_prom:.1f}')
            print(f'  Prom. puntos AZUL:     {pa_prom:.1f}')
            print(f'  Prom. nodos Minimax:   {n_prom:,.0f}')
            print(f'  Prom. profundidad:     {pp_prom:.2f}')
            print(f'  Tiempo total:          {t_total:.1f}s')

        return self.resultados


print(f'{Color.BLANCO}✓ Motor de IA cargado (v2: TT + move ordering + H5 fix + prof media).{Color.RESET}')


## Configuración y Lanzamiento

In [ ]:
# ── Helpers de entrada ────────────────────────────────────────────────────────

def _pedir_int(mensaje: str, defecto: int, minimo: int = 0) -> int:
    try:
        val = int(input(mensaje).strip() or str(defecto))
        return max(minimo, val)
    except ValueError:
        return defecto


def _pedir_float(mensaje: str, defecto: float, minimo: float = 0.1) -> float:
    try:
        val = float(input(mensaje).strip() or str(defecto))
        return max(minimo, val)
    except ValueError:
        return defecto


# ── Selector de agente ────────────────────────────────────────────────────────

def _elegir_agente(nombre_color: str, jugador: Jugador,
                   config_h: ConfiguracionHeuristica,
                   tiempo: float) -> Agente:
    """Muestra el menú de perfiles y retorna el agente elegido."""
    opciones = {
        '1': ('Humano',        lambda: AgenteHumano(jugador)),
        '2': ('Aleatorio',     lambda: AgenteAleatorio(jugador)),
        '3': ('Greedy',        lambda: AgenteGreedy(jugador, config_h)),
        '4': ('PeorDecision',  lambda: AgentePeorDecision(jugador, config_h)),
        '5': ('Minimax',       lambda: AgenteMinimax(jugador, config_h, tiempo,
                                                      nombre='Minimax')),
        '6': ('MinimaxEspejo', lambda: AgenteMinimax(jugador, config_h, tiempo,
                                                      nombre='MinimaxEspejo')),
    }
    rst = Color.RESET
    print(f'\n{nombre_color}┌─ Jugador {jugador.value} ────────────────────────────┐{rst}')
    for k, (nombre, _) in opciones.items():
        print(f'{nombre_color}│{rst}  {k}. {nombre:<24}{nombre_color}      │{rst}')
    print(f'{nombre_color}└──────────────────────────────────────────────┘{rst}')

    while True:
        sel = input(f'  Elige tipo [{jugador.value}] (1-6, por defecto 1): ').strip() or '1'
        if sel in opciones:
            nombre, constructor = opciones[sel]
            print(f'  → {nombre_color}{nombre}{rst} asignado a {jugador.value}')
            return constructor()
        print('  Opción inválida. Ingresa un número entre 1 y 6.')


# ── Configuración de heurísticas ──────────────────────────────────────────────

def _configurar_heuristicas() -> ConfiguracionHeuristica:
    """Permite elegir entre presets o configuración manual de heurísticas."""
    rst = Color.RESET
    bco = Color.BLANCO
    print(f'\n{bco}╔══ CONFIGURACIÓN DE HEURÍSTICAS ══════════════════╗{rst}')
    print(f'{bco}║{rst}  1. Config-1 — Equilibrada (recomendada)         {bco}║{rst}')
    print(f'{bco}║{rst}     H1×1.5 | H2×0.5 | H3×0.3 | H4×1.2 | H5×1.8 {bco}║{rst}')
    print(f'{bco}║{rst}  2. Config-2 — Agresiva (maximiza amenaza)        {bco}║{rst}')
    print(f'{bco}║{rst}     H1×0.5 | H2×0.3 | H3×0.2 | H4×0.6 | H5×3.5 {bco}║{rst}')
    print(f'{bco}║{rst}  3. Manual — elige heurísticas y pesos             {bco}║{rst}')
    print(f'{bco}╚══════════════════════════════════════════════════╝{rst}')

    sel = input('  Elige configuración (1/2/3, por defecto 1): ').strip() or '1'

    if sel == '2':
        print('  → Config-2 Agresiva seleccionada')
        return ConfiguracionHeuristica(pesos=dict(PESOS_CONFIG_2))

    if sel == '3':
        activas: set = set()
        pesos:   dict = {}
        print('\n  Heurísticas disponibles:')
        for idx, (nombre, _) in HEURISTICAS_DISPONIBLES.items():
            print(f'    {idx}. {nombre}')
        ids_raw = input('  ¿Cuáles activar? (ej: 1,2,4 — enter=todas): ').strip()
        if ids_raw:
            for tok in ids_raw.split(','):
                try:
                    activas.add(int(tok.strip()))
                except ValueError:
                    pass
        else:
            activas = {1, 2, 3, 4, 5}
        print('  Pesos (enter = 1.0 por defecto):')
        for idx in sorted(activas):
            nombre = HEURISTICAS_DISPONIBLES[idx][0]
            peso   = _pedir_float(f'    Peso para {nombre}: ', defecto=1.0)
            pesos[idx] = peso
        return ConfiguracionHeuristica(activas=activas, pesos=pesos)

    print('  → Config-1 Equilibrada seleccionada')
    return ConfiguracionHeuristica(pesos=dict(PESOS_CONFIG_1))


# ── Flujo de configuración interactivo ───────────────────────────────────────

def configurar_partida_interactivo() -> ConfiguracionPartida:
    """
    Guía al usuario paso a paso para configurar una partida o serie de partidas.
    Retorna un ConfiguracionPartida listo para pasar a ControladorPartida.
    """
    bco = Color.BLANCO
    rst = Color.RESET

    print(f'\n{bco}{"═"*52}')
    print(f'  ⚔  LASER CHESS — Configuración de partida  ⚔')
    print(f'{"═"*52}{rst}\n')

    # 1. Tiempo límite por turno de IA
    tiempo = _pedir_float(
        '  Tiempo límite por turno de IA en segundos (por defecto 3.0): ',
        defecto=3.0, minimo=0.5,
    )
    print(f'  → {tiempo:.1f}s por turno de IA\n')

    # 2. Configuración de heurísticas
    config_h = _configurar_heuristicas()
    print(f'\n{bco}  Heurísticas activas:{rst}')
    print(config_h.descripcion())

    # 3. Agente ROJO
    agente_rojo = _elegir_agente(Color.ROJO, Jugador.ROJO, config_h, tiempo)

    # 4. Agente AZUL
    agente_azul = _elegir_agente(Color.AZUL, Jugador.AZUL, config_h, tiempo)

    # 5. Número de partidas
    num = _pedir_int(
        '\n  ¿Cuántas partidas jugar? (por defecto 1): ',
        defecto=1, minimo=1,
    )

    # 6. Pausa entre turnos (solo relevante en IA vs IA)
    pausa   = 0
    ambos_ia = (not isinstance(agente_rojo, AgenteHumano)
                and not isinstance(agente_azul, AgenteHumano))
    if ambos_ia:
        pausa = _pedir_int(
            '  Pausa entre turnos en ms (0=sin pausa, por defecto 500): ',
            defecto=500, minimo=0,
        )

    # 7. Mostrar tablero (en benchmarks puede desactivarse para ir más rápido)
    mostrar = True
    if num > 1 and ambos_ia:
        sel = input('  ¿Mostrar tablero? (s/n, por defecto s): ').strip().lower()
        mostrar = sel != 'n'

    # 8. Resumen antes de comenzar
    print(f'\n{bco}{"─"*52}')
    print('  RESUMEN DE CONFIGURACIÓN')
    print(f'{"─"*52}{rst}')
    print(f'  {Color.ROJO}ROJO:{rst} {agente_rojo.nombre}')
    print(f'  {Color.AZUL}AZUL:{rst} {agente_azul.nombre}')
    print(f'  Partidas:  {num}')
    print(f'  Tiempo IA: {tiempo:.1f}s')
    print(f'  Mostrar:   {"Sí" if mostrar else "No"}')
    if ambos_ia:
        print(f'  Pausa:     {pausa}ms entre turnos')
    print(f'{bco}{"─"*52}{rst}\n')
    input('  Presiona ENTER para comenzar...')

    return ConfiguracionPartida(
        agente_rojo     = agente_rojo,
        agente_azul     = agente_azul,
        num_partidas    = num,
        mostrar_tablero = mostrar,
        pausa_ms        = pausa,
    )

### Modos de inicio

In [ ]:
# ── OPCIÓN A: Configuración interactiva completa ──────────────────────────────
# Descomenta las dos líneas siguientes para configurar todo paso a paso.
#
try:
    config = configurar_partida_interactivo()
    ControladorPartida(config).ejecutar()
except KeyboardInterrupt:
    print('\nPartida cancelada. El benchmark puede ejecutarse en la celda siguiente.')


# ── OPCIÓN B (activa por defecto): Humano ROJO vs Minimax AZUL ───────────────
# Tú juegas como ROJO; la IA Minimax (Config-1 equilibrada) juega como AZUL.

# config_h_b = ConfiguracionHeuristica(pesos=dict(PESOS_CONFIG_1))
# config = ConfiguracionPartida(
#     agente_rojo = AgenteHumano(Jugador.ROJO),
#     agente_azul = AgenteMinimax(Jugador.AZUL, config_h_b, tiempo_limite=3.0),
# )
# ControladorPartida(config).ejecutar()


# ── OPCIÓN C: Minimax vs Aleatorio — 3 partidas (benchmark rápido) ───────────
# config_h_c = ConfiguracionHeuristica(pesos=dict(PESOS_CONFIG_1))
# config = ConfiguracionPartida(
#     agente_rojo     = AgenteMinimax(Jugador.ROJO, config_h_c, tiempo_limite=2.0),
#     agente_azul     = AgenteAleatorio(Jugador.AZUL),
#     num_partidas    = 3,
#     mostrar_tablero = True,
#     pausa_ms        = 300,
# )
# ControladorPartida(config).ejecutar()


# ── OPCIÓN D: Minimax (Config-1) vs MinimaxEspejo (Config-2) — 5 partidas ────
# config_h_d1 = ConfiguracionHeuristica(pesos=dict(PESOS_CONFIG_1))
# config_h_d2 = ConfiguracionHeuristica(pesos=dict(PESOS_CONFIG_2))
# config = ConfiguracionPartida(
#     agente_rojo     = AgenteMinimax(Jugador.ROJO, config_h_d1, tiempo_limite=3.0,
#                                      nombre='Minimax'),
#     agente_azul     = AgenteMinimax(Jugador.AZUL, config_h_d2, tiempo_limite=3.0,
#                                      nombre='MinimaxEspejo'),
#     num_partidas    = 5,
#     mostrar_tablero = False,   # sin display para benchmark rápido
#     pausa_ms        = 0,
# )

## Benchmark

In [ ]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
import os

# ── Parámetros ────────────────────────────────────────────────────────────────
GAMES_PER_CONFIG  = 3   # partidas por (heurísticas × tiempo) para rivales normales
GAMES_IA_VS_IA    = 2   # reducido porque ambos agentes piensan (más lento)
MAX_JUGADAS_BM    = 150

# ── Conjuntos de heurísticas evaluados (H1, H1+H2+H3, H1-H5) ─────────────────
# Se prueban 1, 3 y 5 heurísticas para cubrir el espectro sin ejecutar las 5
# combinaciones completas. Las 5 funciones de heurística permanecen intactas.
HEURISTIC_SETS = [
    ('H1',     {1},         'H1 (solo Material)'),
    ('H1+H2+H3', {1,2,3},  'H1+H2+H3'),
    ('H1-H5',  {1,2,3,4,5}, 'H1+H2+H3+H4+H5 (todas)'),
]

# ── Tiempos límite (requeridos por el proyecto) ───────────────────────────────
TIME_LIMITS = [1.0, 3.0, 10.0]

# ── Pesos fijos para todas las pruebas ───────────────────────────────────────
PESOS_BM = PESOS_CONFIG_1

# ── Ruta del archivo de resultados ───────────────────────────────────────────
RUTA_XLSX = 'benchmark_laser_chess.xlsx'

# ── Definición de oponentes ───────────────────────────────────────────────────
def _make_greedy(cfg, t):
    rival_cfg = ConfiguracionHeuristica(activas={1,2,3,4,5}, pesos=dict(PESOS_CONFIG_1))
    return AgenteGreedy(Jugador.AZUL, rival_cfg)

def _make_peor(cfg, t):
    rival_cfg = ConfiguracionHeuristica(activas={1,2,3,4,5}, pesos=dict(PESOS_CONFIG_1))
    return AgentePeorDecision(Jugador.AZUL, rival_cfg)

def _make_ia(cfg, t):
    rival_cfg = ConfiguracionHeuristica(activas=set(cfg.activas), pesos=dict(cfg.pesos))
    return AgenteMinimax(Jugador.AZUL, rival_cfg, tiempo_limite=t, profundidad_max=5)

OPPONENTS = [
    {
        'nombre':      'Aleatorio',
        'clase':       'AgenteAleatorio',
        'color_bg':    'DBEAFE',
        'color_hdr':   '1D4ED8',
        'automatable': True,
        'make':        lambda cfg, t: AgenteAleatorio(Jugador.AZUL),
        'n_games':     GAMES_PER_CONFIG,
    },
    {
        'nombre':      'Greedy',
        'clase':       'AgenteGreedy',
        'color_bg':    'DCFCE7',
        'color_hdr':   '15803D',
        'automatable': True,
        'make':        _make_greedy,
        'n_games':     GAMES_PER_CONFIG,
    },
    {
        'nombre':      'Peor Jugador',
        'clase':       'AgentePeorDecision',
        'color_bg':    'FEF3C7',
        'color_hdr':   '92400E',
        'automatable': True,
        'make':        _make_peor,
        'n_games':     GAMES_PER_CONFIG,
    },
    {
        'nombre':      'Humano Experto',
        'clase':       'AgenteHumano',
        'color_bg':    'F3E8FF',
        'color_hdr':   '6B21A8',
        'automatable': False,
        'make':        None,
        'n_games':     0,
    },
    {
        'nombre':      'IA vs IA',
        'clase':       'AgenteMinimax (mismo config)',
        'color_bg':    'FCE7F3',
        'color_hdr':   '9D174D',
        'automatable': True,
        'make':        _make_ia,
        'n_games':     GAMES_IA_VS_IA,
    },
]


# ═══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN CENTRAL: ejecutar una configuración (heurísticas × tiempo) N partidas
# ═══════════════════════════════════════════════════════════════════════════════
def _run_config(hset_label, hset, t_lim, make_rival, n_games):
    """
    Ejecuta n_games partidas de Minimax(hset, t_lim) contra make_rival(cfg, t_lim).
    Devuelve lista de dicts con métricas por partida.
    """
    config_h = ConfiguracionHeuristica(activas=set(hset), pesos=dict(PESOS_BM))
    filas    = []
    for n in range(1, n_games + 1):
        agente_mm  = AgenteMinimax(Jugador.ROJO, config_h,
                                   tiempo_limite=t_lim, profundidad_max=5)
        agente_riv = make_rival(config_h, t_lim)
        ctrl = ControladorPartida(ConfiguracionPartida(
            agente_rojo     = agente_mm,
            agente_azul     = agente_riv,
            num_partidas    = 1,
            max_jugadas     = MAX_JUGADAS_BM,
            mostrar_tablero = False,
            pausa_ms        = 0,
        ))
        r   = ctrl.jugar_partida()
        gan = r.ganador.value if r.ganador else 'Empate'
        t_por_mov = round(r.duracion_s / max(1, agente_mm._turnos), 3)
        filas.append({
            'hset_label':    hset_label,
            'n_heuristicas': len(hset),
            't_lim':         t_lim,
            'partida':       n,
            'ganador':       gan,
            'pts_rojo':      r.puntos_rojo,
            'pts_azul':      r.puntos_azul,
            'nodos':         r.nodos_expandidos,
            'profundidad':   agente_mm.ultima_profundidad,
            'prof_promedio': round(r.prof_promedio, 2),
            'tiempo_s':      round(r.duracion_s, 2),
            'jugadas':       r.jugadas,
            't_por_mov':     t_por_mov,
        })
        print(f'    [{n}/{n_games}] {gan} | {r.jugadas} jug | '
              f'nodos={r.nodos_expandidos:,} | prof={r.prof_promedio:.1f} | '
              f't/mov={t_por_mov:.2f}s')
    return filas


def _stats(filas):
    """Agrega métricas de una lista de partidas."""
    if not filas:
        return {}
    n = len(filas)
    return {
        'partidas':    n,
        'vic_mm':      sum(1 for f in filas if f['ganador'] == 'ROJO'),
        'vic_rival':   sum(1 for f in filas if f['ganador'] == 'AZUL'),
        'empates':     sum(1 for f in filas if f['ganador'] == 'Empate'),
        'win_rate':    sum(1 for f in filas if f['ganador'] == 'ROJO') / n,
        'prom_pts_r':  sum(f['pts_rojo']      for f in filas) / n,
        'prom_pts_a':  sum(f['pts_azul']      for f in filas) / n,
        'prom_nodos':  sum(f['nodos']         for f in filas) / n,
        'prom_prof':   sum(f['prof_promedio'] for f in filas) / n,
        'prom_tiempo': sum(f['tiempo_s']      for f in filas) / n,
        'prom_t_mov':  sum(f['t_por_mov']     for f in filas) / n,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS PARA EL EXCEL (sin cambios respecto a la versión anterior)
# ═══════════════════════════════════════════════════════════════════════════════
def _thin():
    s = Side(style='thin', color='CCCCCC')
    return Border(left=s, right=s, top=s, bottom=s)

def _hdr(ws, row, col, value, bg='1F3864', fg='FFFFFF', size=10):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = Font(bold=True, color=fg, name='Arial', size=size)
    c.fill      = PatternFill('solid', start_color=bg)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c.border    = _thin()
    return c

def _dat(ws, row, col, value, bold=False, fmt=None, bg=None, fg='000000'):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = Font(name='Arial', size=10, bold=bold, color=fg)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border    = _thin()
    if fmt: c.number_format = fmt
    if bg:  c.fill = PatternFill('solid', start_color=bg)
    return c

def _color_gan(gan):
    if gan == 'ROJO':   return ('FF8080', 'FFFFFF')
    if gan == 'AZUL':   return ('4A90D9', 'FFFFFF')
    return ('D0D0D0', '000000')

RAW_COLS   = ['Heurísticas', 'T.Lim', 'Partida', 'Ganador',
               'Pts ROJO', 'Pts AZUL', 'Nodos', 'Prof.', 'Prof.Prom',
               'Tiempo(s)', 'Jugadas', 'T/Mov(s)']
RAW_WIDTHS = [16, 8, 8, 9, 9, 9, 13, 8, 10, 10, 9, 10]
RAW_KEYS   = ['hset_label', 't_lim', 'partida', 'ganador',
               'pts_rojo', 'pts_azul', 'nodos', 'profundidad', 'prof_promedio',
               'tiempo_s', 'jugadas', 't_por_mov']
RAW_FMTS   = [None, '0.0', None, None,
               None, None, '#,##0', None, '0.00',
               '0.00', None, '0.00']

SUM_COLS   = ['Heurísticas', 'T.Lim', 'Part.', 'Vic.MM', 'Vic.Rival',
               'Empates', 'Win Rate', 'Pts ROJO', 'Pts AZUL',
               'Nodos', 'Prof.', 'Tiempo', 'T/Mov']
SUM_WIDTHS = [16, 8, 6, 8, 9, 8, 9, 9, 9, 13, 8, 10, 9]
SUM_FMTS   = [None, '0.0', None, None, None,
               None, '0.0%', '#,##0.0', '#,##0.0',
               '#,##0', '0.00', '0.00', '0.00']


def _create_opponent_sheet(wb, opp, all_results):
    """Crea una hoja completa para un oponente (con los resultados disponibles hasta ahora)."""
    nombre   = opp['nombre']
    color_bg = opp['color_bg']
    color_hd = opp['color_hdr']
    auto     = opp['automatable']

    ws = wb.create_sheet(nombre)
    ws.sheet_view.showGridLines = False
    ws.freeze_panes = 'A3'

    t = ws.cell(row=1, column=1,
                value=f'MINIMAX vs {nombre.upper()}  ({opp["clase"]})')
    t.font = Font(name='Arial', size=13, bold=True, color=color_hd)
    ws.merge_cells(f'A1:{get_column_letter(len(RAW_COLS))}1')
    t.fill = PatternFill('solid', start_color=color_bg)
    ws.row_dimensions[1].height = 28

    if not auto:
        msg = ws.cell(row=3, column=1,
                      value='⚠  Este oponente requiere partidas manuales. '
                            'Ejecuta cell-5 → opción Humano vs IA y registra '
                            'los resultados en esta hoja con el mismo formato '
                            'que las demás hojas.')
        msg.font = Font(name='Arial', size=11, italic=True, color='6B21A8')
        ws.merge_cells(f'A3:{get_column_letter(len(RAW_COLS))}3')
        msg.fill = PatternFill('solid', start_color=color_bg)
        for i, (h, w) in enumerate(zip(RAW_COLS, RAW_WIDTHS), 1):
            _hdr(ws, 5, i, h, bg=color_hd)
            ws.column_dimensions[get_column_letter(i)].width = w
        ws.row_dimensions[5].height = 26
        return

    for i, (h, w) in enumerate(zip(RAW_COLS, RAW_WIDTHS), 1):
        _hdr(ws, 2, i, h, bg=color_hd)
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.row_dimensions[2].height = 26

    raw_row = 3
    prev_hset = None
    for hset_label, hset, _ in HEURISTIC_SETS:
        for t_lim in TIME_LIMITS:
            filas = all_results.get((hset_label, t_lim), [])
            if hset_label != prev_hset and prev_hset is not None:
                for ci in range(1, len(RAW_COLS) + 1):
                    ws.cell(row=raw_row, column=ci).fill = \
                        PatternFill('solid', start_color='F1F5F9')
                    ws.cell(row=raw_row, column=ci).border = _thin()
                ws.row_dimensions[raw_row].height = 6
                raw_row += 1
            prev_hset = hset_label

            for fila in filas:
                bg_g, fg_g = _color_gan(fila['ganador'])
                group_bg   = color_bg if fila['partida'] % 2 == 1 else 'FFFFFF'
                for ci, (key, fmt) in enumerate(zip(RAW_KEYS, RAW_FMTS), 1):
                    c = _dat(ws, raw_row, ci, fila[key], fmt=fmt, bg=group_bg)
                    if key == 'hset_label':
                        c.font = Font(name='Arial', size=10, bold=True, color=color_hd)
                    if key == 'ganador':
                        c.fill = PatternFill('solid', start_color=bg_g)
                        c.font = Font(name='Arial', size=10, bold=True, color=fg_g)
                raw_row += 1

    raw_row += 1
    sum_title = ws.cell(row=raw_row, column=1,
                        value=f'RESUMEN — Minimax vs {nombre}')
    sum_title.font = Font(name='Arial', size=11, bold=True, color=color_hd)
    sum_title.fill = PatternFill('solid', start_color=color_bg)
    ws.merge_cells(f'A{raw_row}:{get_column_letter(len(SUM_COLS))}{raw_row}')
    ws.row_dimensions[raw_row].height = 22
    raw_row += 1

    for i, (h, w) in enumerate(zip(SUM_COLS, SUM_WIDTHS), 1):
        _hdr(ws, raw_row, i, h, bg=color_hd)
        ws.column_dimensions[get_column_letter(i)].width = max(
            ws.column_dimensions[get_column_letter(i)].width, w)
    ws.row_dimensions[raw_row].height = 26
    raw_row += 1

    alt = False
    for hset_label, hset, _ in HEURISTIC_SETS:
        for t_lim in TIME_LIMITS:
            filas = all_results.get((hset_label, t_lim), [])
            s     = _stats(filas)
            bg    = color_bg if alt else 'FFFFFF'
            if s:
                vals = [hset_label, t_lim,
                        s['partidas'], s['vic_mm'], s['vic_rival'],
                        s['empates'], s['win_rate'],
                        s['prom_pts_r'], s['prom_pts_a'],
                        s['prom_nodos'], s['prom_prof'],
                        s['prom_tiempo'], s['prom_t_mov']]
            else:
                vals = [hset_label, t_lim] + ['—'] * (len(SUM_COLS) - 2)
            for ci, (v, fmt) in enumerate(zip(vals, SUM_FMTS), 1):
                c = _dat(ws, raw_row, ci, v, fmt=fmt, bg=bg)
                if ci == 1:
                    c.font = Font(name='Arial', size=10, bold=True, color=color_hd)
            raw_row += 1
            alt = not alt


def _save_excel(results):
    """
    Construye y guarda el workbook completo con los resultados disponibles hasta ahora.
    Se llama después de cada configuración para guardar progresivamente.
    Las configuraciones pendientes aparecen con '—' hasta completarse.
    """
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    for opp in OPPONENTS:
        _create_opponent_sheet(wb, opp, results.get(opp['nombre'], {}))

    # Hoja de índice
    ws_idx = wb.create_sheet('Índice', 0)
    ws_idx.sheet_view.showGridLines = False
    ws_idx.column_dimensions['A'].width = 20
    ws_idx.column_dimensions['B'].width = 30
    ws_idx.column_dimensions['C'].width = 22
    ws_idx.column_dimensions['D'].width = 16

    t = ws_idx.cell(row=1, column=1, value='LASER CHESS — BENCHMARK')
    t.font = Font(name='Arial', size=15, bold=True, color='1F3864')
    ws_idx.merge_cells('A1:D1')
    t.fill = PatternFill('solid', start_color='EFF6FF')
    ws_idx.row_dimensions[1].height = 30

    ws_idx.cell(row=2, column=1).value = f'Actualizado: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
    ws_idx.cell(row=2, column=1).font  = Font(name='Arial', size=10, italic=True, color='64748B')
    ws_idx.merge_cells('A2:D2')

    _hdr(ws_idx, 4, 1, 'Hoja',        bg='334155')
    _hdr(ws_idx, 4, 2, 'Oponente',    bg='334155')
    _hdr(ws_idx, 4, 3, 'Clase rival', bg='334155')
    _hdr(ws_idx, 4, 4, 'Partidas',    bg='334155')

    for ri, opp in enumerate(OPPONENTS, 5):
        nombre  = opp['nombre']
        n_games = opp['n_games']
        total_p = len(HEURISTIC_SETS) * len(TIME_LIMITS) * n_games if opp['automatable'] else 0
        bg = opp['color_bg']
        _dat(ws_idx, ri, 1, nombre,        bold=True, bg=bg)
        _dat(ws_idx, ri, 2, nombre,        bg=bg)
        _dat(ws_idx, ri, 3, opp['clase'],  bg=bg)
        _dat(ws_idx, ri, 4,
             total_p if opp['automatable'] else 'Manual',
             bg=bg)

    info_row = 5 + len(OPPONENTS) + 2
    ws_idx.cell(row=info_row, column=1).value = 'CONFIGURACIONES EVALUADAS'
    ws_idx.cell(row=info_row, column=1).font  = Font(name='Arial', size=11, bold=True, color='1D4ED8')
    ws_idx.merge_cells(f'A{info_row}:D{info_row}')
    ws_idx.cell(row=info_row, column=1).fill = PatternFill('solid', start_color='EFF6FF')

    for ri, (hset_label, hset, hset_desc) in enumerate(HEURISTIC_SETS, info_row + 1):
        ws_idx.cell(row=ri, column=1).value = hset_label
        ws_idx.cell(row=ri, column=1).font  = Font(name='Arial', size=10, bold=True)
        ws_idx.cell(row=ri, column=2).value = hset_desc
        ws_idx.cell(row=ri, column=2).font  = Font(name='Arial', size=10)
        ws_idx.cell(row=ri, column=3).value = f'T límite: {", ".join(str(tl)+"s" for tl in TIME_LIMITS)}'
        ws_idx.cell(row=ri, column=3).font  = Font(name='Arial', size=10, color='64748B')

    wb.save(RUTA_XLSX)


# ═══════════════════════════════════════════════════════════════════════════════
# BUCLE PRINCIPAL DEL BENCHMARK
# ═══════════════════════════════════════════════════════════════════════════════
total_configs = len(HEURISTIC_SETS) * len(TIME_LIMITS)
print('═' * 70)
print('  LASER CHESS — BENCHMARK POR TIPO DE OPONENTE')
print(f'  {len(OPPONENTS)} oponentes  ×  {len(HEURISTIC_SETS)} conjuntos heurísticas'
      f'  ×  {len(TIME_LIMITS)} tiempos  |  prof. máx = 5')
print(f'  Partidas normales: {GAMES_PER_CONFIG}/config  |  IA vs IA: {GAMES_IA_VS_IA}/config')
print(f'  Guardado progresivo activado → {RUTA_XLSX}')
print('═' * 70)

# results[opp_nombre][(hset_label, t_lim)] = lista de filas
results = {}

for opp in OPPONENTS:
    opp_nombre = opp['nombre']
    results[opp_nombre] = {}

    print(f'\n{"═"*70}')
    print(f'  OPONENTE: {opp_nombre}  ({opp["clase"]})')
    if not opp['automatable']:
        print(f'  ⚠  No automatizable — hoja creada con estructura vacía para llenado manual.')
        print(f'{"═"*70}')
        _save_excel(results)   # guardar hoja vacía del humano también
        continue
    print(f'{"═"*70}')

    for hset_label, hset, hset_desc in HEURISTIC_SETS:
        for t_lim in TIME_LIMITS:
            key = (hset_label, t_lim)
            print(f'\n  [{hset_label}  t={t_lim}s]  {hset_desc}')
            filas = _run_config(hset_label, hset, t_lim,
                                make_rival=opp['make'],
                                n_games=opp['n_games'])
            results[opp_nombre][key] = filas
            # ── Guardado progresivo: actualizar el Excel después de cada config ──
            _save_excel(results)
            print(f'  → Guardado: {RUTA_XLSX}')

    # Tabla resumen en consola para este oponente
    print(f'\n  {"─"*68}')
    print(f'  RESUMEN — Minimax vs {opp_nombre}')
    print(f'  {"─"*68}')
    print(f'  {"Config":<12} {"T":>4}  {"MM":>4}  {"Riv":>4}  {"Emp":>4}'
          f'  {"Win%":>7}  {"PtsR":>5}  {"Nodos":>9}  {"Prof":>5}  {"T/mov":>6}')
    print(f'  {"─"*68}')
    for hset_label, hset, _ in HEURISTIC_SETS:
        for t_lim in TIME_LIMITS:
            s = _stats(results[opp_nombre].get((hset_label, t_lim), []))
            if not s:
                continue
            print(f'  {hset_label:<12} {t_lim:>4.0f}  {s["vic_mm"]:>4}  {s["vic_rival"]:>4}'
                  f'  {s["empates"]:>4}  {s["win_rate"]:>6.0%}  {s["prom_pts_r"]:>5.1f}'
                  f'  {s["prom_nodos"]:>9,.0f}  {s["prom_prof"]:>5.1f}'
                  f'  {s["prom_t_mov"]:>5.2f}s')

total_auto = sum(
    len(HEURISTIC_SETS) * len(TIME_LIMITS) * opp['n_games']
    for opp in OPPONENTS if opp['automatable']
)
print(f'\n✓ Benchmark completado.')
print(f'  Excel: {os.path.abspath(RUTA_XLSX)}')
print(f'  Hojas: {", ".join(opp["nombre"] for opp in OPPONENTS)} + Índice')
print(f'  Total partidas ejecutadas: {total_auto}')
print(f'  Hoja "Humano Experto": estructura vacía para llenado manual')
